# 🎯 Advanced Object Detection with YOLO

Welcome to the Advanced Object Detection tutorial using YOLO (You Only Look Once) models! This notebook demonstrates real-time object detection capabilities using the latest YOLOv8 architecture.

## 📚 What You'll Learn
- YOLO object detection fundamentals
- Real-time object detection
- Multiple object classes
- Confidence scoring and filtering
- Performance optimization
- Custom model training concepts

## 🚀 Prerequisites
- YOLO library installed (`pip install ultralytics`)
- Test images or video files
- Basic Python knowledge

In [ ]:
# Import required libraries
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

# Import ValidoAI Computer Vision
from src.core.computer_vision import create_computer_vision_processor
import matplotlib.pyplot as plt
import numpy as np
import time

# Create CV processor
cv = create_computer_vision_processor()

# Check YOLO availability
print("🔍 Checking YOLO Availability:")
deps = cv.check_dependencies()
if deps.get('yolo', False):
    print("✅ YOLO is available and ready to use!")
else:
    print("❌ YOLO not available. Install with: pip install ultralytics")

## 🎯 YOLO Object Detection Basics

YOLO (You Only Look Once) is a state-of-the-art object detection algorithm known for its speed and accuracy. Let's explore its capabilities.

In [ ]:
# Load test image
test_images = [
    "static/images/sample.jpg",
    "static/images/logo.png",
    "data/sample_image.jpg",
    "data/test_objects.jpg"  # Add images with objects
]

image_path = None
for path in test_images:
    if os.path.exists(path):
        image_path = path
        break

if image_path:
    print(f"📸 Loading image for object detection: {image_path}")
    image = cv.load_image(image_path)
    
    if image is not None:
        print("✅ Image loaded successfully!")
        
        # Get image info
        info = cv.get_image_info(image)
        print(f"📊 Image size: {info.get('width', 'N/A')}x{info.get('height', 'N/A')}")
        
        # Display original image
        try:
            plt.figure(figsize=(12, 8))
            plt.imshow(image)
            plt.title('Original Image for Object Detection')
            plt.axis('off')
            plt.show()
        except:
            print("📊 Image loaded (display not available)")
    else:
        print("❌ Failed to load image")
else:
    print("⚠️  No test image found. Please add an image with objects to test.")

## 🚀 YOLO Object Detection

Let's run YOLO object detection on the loaded image and see what objects it can detect.

In [ ]:
if image is not None and cv.yolo_available:
    print("🎯 Running YOLO Object Detection...")
    print("⏱️  This may take a moment to download the model on first run...")
    
    # Time the detection process
    start_time = time.time()
    
    # Run YOLO detection with different model sizes
    models_to_test = ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt']
    
    for model_name in models_to_test:
        print(f"\n🔍 Testing {model_name}...")
        try:
            detections = cv.object_detection_yolo(image, model_name)
            
            if detections:
                print(f"✅ {model_name}: Detected {len(detections)} objects")
                
                # Show top detections
                for i, detection in enumerate(detections[:10]):  # Show top 10
                    print(f"  {i+1}. {detection['class']} (confidence: {detection['confidence']:.3f})")
                    bbox = detection['bbox']
                    print(f"     Location: ({bbox[0]:.0f}, {bbox[1]:.0f}, {bbox[2]:.0f}, {bbox[3]:.0f})")
                
                break  # Use first successful model
            else:
                print(f"⚠️  {model_name}: No objects detected")
                
        except Exception as e:
            print(f"❌ {model_name}: Error - {e}")
    
    total_time = time.time() - start_time
    print(f"\n⏱️  Total detection time: {total_time:.2f}s")
    
else:
    print("⚠️  YOLO object detection test skipped (no image or YOLO not available)")

## 🎨 Visualizing Detection Results

Let's visualize the detected objects with bounding boxes and labels.

In [ ]:
if image is not None and cv.yolo_available and 'detections' in locals():
    print("🎨 Visualizing Detection Results...")
    
    try:
        # Create a copy of the image for annotation
        annotated_image = image.copy()
        
        # Import OpenCV for drawing
        import cv2
        
        # Draw bounding boxes and labels
        for detection in detections:
            bbox = detection['bbox']
            class_name = detection['class']
            confidence = detection['confidence']
            
            # Convert bbox coordinates to integers
            x1, y1, x2, y2 = map(int, bbox)
            
            # Draw rectangle
            cv2.rectangle(annotated_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            # Prepare label text
            label = f"{class_name}: {confidence:.2f}"
            
            # Draw label background
            (text_width, text_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            cv2.rectangle(annotated_image, (x1, y1 - text_height - 10), (x1 + text_width, y1), (0, 255, 0), -1)
            
            # Draw label text
            cv2.putText(annotated_image, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
        
        # Display annotated image
        plt.figure(figsize=(15, 10))
        plt.imshow(annotated_image)
        plt.title(f'YOLO Object Detection Results ({len(detections)} objects)')
        plt.axis('off')
        plt.show()
        
        # Save annotated image
        output_path = "data/detection_results.jpg"
        if cv.save_image(annotated_image, output_path):
            print(f"💾 Annotated image saved: {output_path}")
        
    except Exception as e:
        print(f"❌ Error creating visualization: {e}")
        
else:
    print("⚠️  Visualization skipped (no detections or image available)")

## 📊 Detection Analysis

Let's analyze the detection results and explore filtering options.

In [ ]:
if 'detections' in locals() and detections:
    print("📊 Analyzing Detection Results...")
    
    # Analyze detection statistics
    classes = [d['class'] for d in detections]
    confidences = [d['confidence'] for d in detections]
    
    # Count objects by class
    from collections import Counter
    class_counts = Counter(classes)
    
    print(f"\n🎯 Detection Statistics:")
    print(f"  Total objects detected: {len(detections)}")
    print(f"  Unique classes: {len(set(classes))}")
    print(f"  Average confidence: {np.mean(confidences):.3f}")
    print(f"  Highest confidence: {np.max(confidences):.3f}")
    print(f"  Lowest confidence: {np.min(confidences):.3f}")
    
    print("\n📋 Objects by Class:")
    for class_name, count in sorted(class_counts.items()):
        avg_conf = np.mean([d['confidence'] for d in detections if d['class'] == class_name])
        print(f"  {class_name}: {count} objects (avg confidence: {avg_conf:.3f})")
    
    # Filter detections by confidence
    confidence_thresholds = [0.3, 0.5, 0.7, 0.9]
    print("\n🔍 Filtering by Confidence:")
    for threshold in confidence_thresholds:
        filtered = [d for d in detections if d['confidence'] >= threshold]
        print(f"  Confidence ≥ {threshold:.1f}: {len(filtered)} objects")
    
    # Analyze bounding box sizes
    bbox_areas = []
    for detection in detections:
        bbox = detection['bbox']
        width = bbox[2] - bbox[0]
        height = bbox[3] - bbox[1]
        area = width * height
        bbox_areas.append(area)
    
    print(f"\n📏 Bounding Box Analysis:")
    print(f"  Average area: {np.mean(bbox_areas):.0f} pixels²")
    print(f"  Largest object: {np.max(bbox_areas):.0f} pixels²")
    print(f"  Smallest object: {np.min(bbox_areas):.0f} pixels²")
    
else:
    print("⚠️  Detection analysis skipped (no detections available)")

## ⚡ Performance Optimization

Let's explore different YOLO models and their performance characteristics.

In [ ]:
if image is not None and cv.yolo_available:
    print("⚡ YOLO Model Performance Comparison...")
    
    # Test different YOLO models for performance
    models_to_test = [
        ('yolov8n.pt', 'Nano - Fastest'),
        ('yolov8s.pt', 'Small - Balanced'),
        ('yolov8m.pt', 'Medium - Better accuracy'),
        ('yolov8l.pt', 'Large - Highest accuracy')
    ]
    
    performance_results = []
    
    for model_name, description in models_to_test:
        print(f"\n🚀 Testing {description} ({model_name})...")
        
        try:
            # Time the model loading and inference
            start_time = time.time()
            
            # Run detection 5 times for averaging
            times = []
            detections_list = []
            
            for _ in range(5):
                run_start = time.time()
                detections = cv.object_detection_yolo(image, model_name)
                run_time = time.time() - run_start
                times.append(run_time)
                detections_list.append(detections)
            
            total_time = time.time() - start_time
            
            # Calculate statistics
            avg_time = np.mean(times)
            std_time = np.std(times)
            detections_per_run = [len(d) for d in detections_list if d]
            avg_detections = np.mean(detections_per_run) if detections_per_run else 0
            
            result = {
                'model': model_name,
                'description': description,
                'avg_inference_time': avg_time,
                'std_inference_time': std_time,
                'total_time': total_time,
                'avg_detections': avg_detections,
                'detections_per_second': avg_detections / avg_time if avg_time > 0 else 0
            }
            
            performance_results.append(result)
            
            print(f"  ⏱️  Average inference: {avg_time:.3f}s (±{std_time:.3f}s)")
            print(f"  🎯 Average detections: {avg_detections:.1f}")
            print(f"  ⚡ Detections/second: {result['detections_per_second']:.1f}")
            
        except Exception as e:
            print(f"  ❌ Error testing {model_name}: {e}")
    
    # Display performance comparison
    if performance_results:
        print("\n📊 Performance Comparison:")
        print("-" * 70)
        print("Model          | Avg Time | Std Dev | Detections | Det/Sec")
        print("-" * 70)
        
        for result in performance_results:
            print("12")
        
        # Find best model for different use cases
        fastest = min(performance_results, key=lambda x: x['avg_inference_time'])
        most_accurate = max(performance_results, key=lambda x: x['avg_detections'])
        
        print("\n🏆 Recommendations:")
        print(f"  🚀 Fastest: {fastest['model']} ({fastest['avg_inference_time']:.3f}s)")
        print(f"  🎯 Most Accurate: {most_accurate['model']} ({most_accurate['avg_detections']:.1f} detections)")
    
else:
    print("⚠️  Performance comparison skipped (no image or YOLO not available)")

## 🎥 Real-Time Processing Concepts

Let's explore concepts for real-time object detection processing.

In [ ]:
print("🎥 Real-Time Object Detection Concepts")
print("=" * 50)

# Simulate real-time processing
if image is not None and cv.yolo_available:
    print("\n🔄 Simulating Real-Time Processing...")
    
    # Process multiple frames (simulate video)
    num_frames = 10
    frame_times = []
    
    print(f"Processing {num_frames} frames...")
    for i in range(num_frames):
        start_time = time.time()
        detections = cv.object_detection_yolo(image, 'yolov8n.pt')
        frame_time = time.time() - start_time
        frame_times.append(frame_time)
        
        # Calculate FPS
        fps = 1.0 / frame_time if frame_time > 0 else 0
        print(f"  Frame {i+1}: {frame_time:.3f}s ({fps:.1f} FPS)")
    
    # Calculate statistics
    avg_fps = np.mean([1.0/t if t > 0 else 0 for t in frame_times])
    min_fps = np.min([1.0/t if t > 0 else 0 for t in frame_times])
    max_fps = np.max([1.0/t if t > 0 else 0 for t in frame_times])
    
    print(f"\n📊 Real-Time Statistics:")
    print(f"  Average FPS: {avg_fps:.1f}")
    print(f"  Min FPS: {min_fps:.1f}")
    print(f"  Max FPS: {max_fps:.1f}")
    print(f"  Real-time capable: {'✅ Yes' if avg_fps >= 30 else '❌ No'} (30 FPS threshold)")
    
else:
    print("⚠️  Real-time simulation skipped (no image or YOLO not available)")

# Display optimization tips
print("\n💡 Real-Time Optimization Tips:")
print("  1. Use YOLOv8n (nano) for fastest inference")
print("  2. Reduce image size for processing")
print("  3. Use confidence thresholds to filter detections")
print("  4. Process frames asynchronously")
print("  5. Use GPU acceleration if available")
print("  6. Implement frame skipping for very fast motion")

## 🛠️ Custom Model Training Concepts

Let's explore the concepts behind training custom YOLO models.

In [ ]:
print("🛠️ Custom YOLO Model Training Concepts")
print("=" * 50)

# Display training concepts
training_concepts = {
    'dataset_preparation': {
        'description': 'Prepare labeled images for training',
        'tools': ['LabelImg', 'CVAT', 'Roboflow', 'Labelbox'],
        'format': 'YOLO format (.txt files with bounding boxes)',
        'requirements': '1000+ images per class for good results'
    },
    'data_augmentation': {
        'techniques': [
            'Random rotation and scaling',
            'Brightness and contrast adjustment',
            'Horizontal/vertical flipping',
            'Mosaic augmentation',
            'Cutout/Mixup'
        ],
        'benefits': 'Improves model generalization and reduces overfitting'
    },
    'model_configuration': {
        'yaml_file': 'Define model architecture and classes',
        'hyperparameters': 'Learning rate, batch size, epochs',
        'pretrained_weights': 'Start from COCO dataset weights',
        'model_sizes': ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt', 'yolov8l.pt']
    },
    'training_process': {
        'command': 'yolo train data=dataset.yaml model=yolov8n.pt epochs=100',
        'monitoring': 'Watch for overfitting, validation loss',
        'early_stopping': 'Stop when validation metrics plateau',
        'hardware': 'GPU recommended for faster training'
    },
    'evaluation_metrics': {
        'mAP': 'Mean Average Precision (primary metric)',
        'precision': 'True positives / (True positives + False positives)',
        'recall': 'True positives / (True positives + False negatives)',
        'F1_score': 'Harmonic mean of precision and recall'
    }
}

for category, details in training_concepts.items():
    print(f"\n🎯 {category.replace('_', ' ').title()}:")
    
    if isinstance(details, dict):
        for key, value in details.items():
            if isinstance(value, list):
                print(f"  📋 {key.replace('_', ' ').title()}:")
                for item in value:
                    print(f"    • {item}")
            else:
                print(f"  📝 {key.replace('_', ' ').title()}: {value}")
    else:
        print(f"  📝 {details}")

# Sample training code (conceptual)
print("\n" + "=" * 50)
print("💻 Sample Training Code (Conceptual):")
print("=" * 50)

sample_code = '''
# Install required packages
!pip install ultralytics

# Import libraries
from ultralytics import YOLO
import os

# Load a pretrained model
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data='path/to/your/dataset.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    name='custom_yolo_model'
)

# Validate the model
metrics = model.val()

# Export the model
model.export(format='onnx')
'''

print(sample_code)

## 🎉 Summary

Congratulations! You've successfully explored advanced object detection with YOLO. Here's what you accomplished:

### ✅ Key Achievements:
1. **YOLO Object Detection**: Successfully detected objects in images
2. **Model Comparison**: Compared different YOLO model sizes and performance
3. **Visualization**: Created annotated images with bounding boxes and labels
4. **Analysis**: Analyzed detection statistics and confidence scores
5. **Performance**: Measured real-time processing capabilities
6. **Training Concepts**: Learned about custom model training

### 🚀 Next Steps:
- **Face Recognition Notebook**: Advanced facial analysis and recognition
- **Medical Imaging Notebook**: Specialized healthcare image analysis
- **Document Analysis Notebook**: OCR and document processing
- **Real-Time Video Notebook**: Video processing and streaming

### 📚 YOLO Model Sizes:
- **YOLOv8n**: Nano - Fastest, least accurate
- **YOLOv8s**: Small - Balanced performance
- **YOLOv8m**: Medium - Better accuracy
- **YOLOv8l**: Large - Highest accuracy
- **YOLOv8x**: Extra Large - Maximum accuracy

### 🎯 Use Cases:
- **Security**: Surveillance and intrusion detection
- **Retail**: Customer behavior analysis
- **Manufacturing**: Quality control and defect detection
- **Healthcare**: Medical imaging analysis
- **Transportation**: Autonomous vehicles and traffic monitoring

YOLO object detection is revolutionizing computer vision with its speed and accuracy! 🎯✨